In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 0 — Montar Drive
# Sube a tu Google Drive:
#   - modelo_condominio_final.pt  (base de entrenamiento)
#   - carpeta 'reales/' con tus fotos/videos del condominio (opcional)
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

RUTA_MODELO_BASE  = '/content/drive/MyDrive/modelo_condominio_final.pt'
RUTA_REALES_DRIVE = '/content/drive/MyDrive/reales'   # fotos/videos reales (opcional)

if Path(RUTA_MODELO_BASE).exists():
    shutil.copy(RUTA_MODELO_BASE, '/content/modelo_condominio_final.pt')
    print('✅ modelo_condominio_final.pt copiado')
else:
    print('⚠️  modelo_condominio_final.pt no encontrado — se usará EfficientNet desde ImageNet')

TIENE_REALES = Path(RUTA_REALES_DRIVE).exists()
print(f'📁 Datos reales en Drive: {TIENE_REALES}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 1 — Configurar Kaggle
#
# Antes de correr:
#   1. Clic en el ícono de llave (🔑) en Colab
#   2. Agregar secret: Name = KAGGLE_TOKEN, Value = tu token KGAT_...
#   3. Activar toggle 'Notebook access'
# ─────────────────────────────────────────────────────────────────────────────
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = 'roly'
os.environ['KAGGLE_TOKEN']    = userdata.get('KAGGLE_TOKEN')

!pip install kaggle -q
print('✅ Kaggle configurado')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 2 — Descargar datasets (más que la versión anterior)
#
# INFRACCION (mal estacionado):
#   - Road Issues Detection    ~9.600 imgs  (versión anterior)
#   - Traffic Violations       ~3.000 imgs  (nuevo)
#
# NORMAL (bien estacionado):
#   - PKLot                   ~12.400 imgs  (versión anterior, ahora SIN límite)
#   - CNRPark                  ~7.000 imgs  (nuevo — cámara de seguridad real)
# ─────────────────────────────────────────────────────────────────────────────

print('📥 [1/4] Road Issues Detection...')
!kaggle datasets download -d programmerrdai/road-issues-detection-dataset -p /content/road_issues --unzip -q
print('✅ Road Issues descargado')

print('\n📥 [2/4] PKLot (parqueos desde cámara de seguridad)...')
!kaggle datasets download -d ammarnassanalhajali/pklot-dataset -p /content/pklot --unzip -q
print('✅ PKLot descargado')

print('\n📥 [3/4] Traffic Violations Dataset...')
!kaggle datasets download -d dataclusterlabs/traffic-violations-images -p /content/traffic_violations --unzip -q 2>/dev/null || \
 kaggle datasets download -d iphysresearch/traffic-violations -p /content/traffic_violations --unzip -q 2>/dev/null || \
 echo '⚠️  Traffic violations no disponible — continuando sin él'

print('\n📥 [4/4] Parking Lot Empty/Occupied...')
!kaggle datasets download -d brendanhasz/parking-lot-images -p /content/parking_extra --unzip -q 2>/dev/null || \
 echo '⚠️  Dataset extra no disponible — continuando sin él'

print('\n✅ Descarga completada')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 3 — Extraer frames de videos reales (si los tienes en Drive)
#
# Sube a Drive/reales/ tus videos (.mp4/.avi) y fotos (.jpg/.png)
# separados en subcarpetas:  reales/infraccion/  y  reales/normal/
#
# De cada video extrae 1 frame cada 30 (~1 frame/segundo a 30fps)
# Un video de 2 minutos → ~120 imágenes reales del condominio
# ─────────────────────────────────────────────────────────────────────────────
import cv2
from pathlib import Path

EXTENSIONES_VIDEO  = {'.mp4', '.avi', '.mov', '.mkv'}
EXTENSIONES_IMAGEN = {'.jpg', '.jpeg', '.png'}
FRAME_CADA_N       = 30   # extraer 1 frame cada N frames del video

def extraer_frames_video(ruta_video, carpeta_destino, clase):
    cap = cv2.VideoCapture(str(ruta_video))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    extraidos = 0
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_CADA_N == 0:
            nombre = carpeta_destino / f'real_{clase}_{ruta_video.stem}_{frame_idx:06d}.jpg'
            cv2.imwrite(str(nombre), frame)
            extraidos += 1
        frame_idx += 1
    cap.release()
    return extraidos

total_reales = 0

if TIENE_REALES:
    for clase in ['infraccion', 'normal']:
        src = Path(RUTA_REALES_DRIVE) / clase
        if not src.exists():
            print(f'⚠️  No existe Drive/reales/{clase}/ — saltando')
            continue

        dst = Path(f'/content/reales/{clase}')
        dst.mkdir(parents=True, exist_ok=True)

        # Copiar imágenes directamente
        for img in src.rglob('*'):
            if img.suffix.lower() in EXTENSIONES_IMAGEN:
                shutil.copy(img, dst / img.name)
                total_reales += 1

        # Extraer frames de videos
        for vid in src.rglob('*'):
            if vid.suffix.lower() in EXTENSIONES_VIDEO:
                n = extraer_frames_video(vid, dst, clase)
                print(f'  🎬 {vid.name} → {n} frames extraídos')
                total_reales += n

    print(f'\n✅ Total imágenes reales procesadas: {total_reales}')
else:
    print('ℹ️  Sin datos reales — entrenando solo con datasets de Kaggle')
    print('   Para agregar tus fotos/videos: crea Drive/reales/infraccion/ y Drive/reales/normal/')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 4 — Organizar dataset (SIN límite de imágenes)
#
# Versión anterior: máximo 3.000 imgs/clase
# Esta versión: usa TODAS las imágenes disponibles
#
# Las imágenes reales del condominio se duplican (x3) para darles
# más peso en el entrenamiento aunque sean pocas.
# ─────────────────────────────────────────────────────────────────────────────
import random
import shutil
from pathlib import Path

random.seed(42)
SPLIT_VAL   = 0.20
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
PESO_REALES = 3   # repetir imágenes reales N veces para darles más peso

def recopilar_imagenes(dirs):
    imgs = []
    for d in dirs:
        p = Path(d)
        if p.exists():
            imgs += [f for f in p.rglob('*') if f.suffix.lower() in EXTENSIONES]
    return imgs

def organizar_clase(origen_dirs, clase, reales_dir=None):
    imagenes = recopilar_imagenes(origen_dirs)
    random.shuffle(imagenes)

    # Agregar imágenes reales con peso extra
    if reales_dir and Path(reales_dir).exists():
        reales = recopilar_imagenes([reales_dir])
        imagenes += reales * PESO_REALES
        print(f'  + {len(reales)} imágenes reales x{PESO_REALES} = {len(reales)*PESO_REALES} extra')

    random.shuffle(imagenes)
    split_idx = int(len(imagenes) * (1 - SPLIT_VAL))
    splits = [('train', imagenes[:split_idx]), ('val', imagenes[split_idx:])]

    for split_name, imgs in splits:
        dst = Path(f'/content/dataset/{split_name}/{clase}')
        dst.mkdir(parents=True, exist_ok=True)
        for i, img in enumerate(imgs):
            try:
                shutil.copy(img, dst / f'{clase}_{i:06d}.jpg')
            except Exception:
                pass
        print(f'  {split_name}/{clase}: {len(imgs)} imágenes')

print('Organizando clase: infraccion...')
organizar_clase(
    origen_dirs=['/content/road_issues', '/content/traffic_violations'],
    clase='infraccion',
    reales_dir='/content/reales/infraccion'
)

print('\nOrganizando clase: normal...')
organizar_clase(
    origen_dirs=['/content/pklot', '/content/parking_extra'],
    clase='normal',
    reales_dir='/content/reales/normal'
)

print('\n=== RESUMEN DATASET ===')
for split in ['train', 'val']:
    for clase in ['infraccion', 'normal']:
        n = len(list(Path(f'/content/dataset/{split}/{clase}').glob('*')))
        print(f'  {split}/{clase}: {n} imágenes')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 5 — DataLoaders con augmentation mejorado
#
# Mejoras vs versión anterior:
#   - RandomPerspective: simula distintos ángulos de cámara
#   - ColorJitter más agresivo: simula noche, lluvia, distintas iluminaciones
#   - RandomGrayscale: simula cámaras IR/nocturnas
#   - GaussianBlur: simula cámara desenfocada o lluvia
# ─────────────────────────────────────────────────────────────────────────────
import torch
from torchvision import datasets, transforms

transform_train = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(15),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.4),
    transforms.ColorJitter(
        brightness=(0.2, 1.5),   # simula noche (0.2) hasta sobreexposición (1.5)
        contrast=(0.5, 1.5),
        saturation=(0.3, 1.5),
        hue=0.05
    ),
    transforms.RandomGrayscale(p=0.15),      # simula cámara IR nocturna
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),  # lluvia/desenfoque
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),  # oclusión parcial
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder('/content/dataset/train', transform=transform_train)
val_ds   = datasets.ImageFolder('/content/dataset/val',   transform=transform_val)

print(f'Clases: {train_ds.classes}')   # ['infraccion', 'normal']
print(f'Train: {len(train_ds)} imágenes')
print(f'Val:   {len(val_ds)} imágenes')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 6 — Modelo: EfficientNet-B0 (mejor que ResNet-18)
#
# EfficientNet-B0 vs ResNet-18:
#   - Similar velocidad de inferencia
#   - ~5% más preciso en clasificación de imágenes
#   - Mejor generalización con datasets pequeños
#
# Pipeline:
#   EfficientNet-B0 (ImageNet) → fine-tune → 2 clases (infraccion/normal)
#
# Si modelo_condominio_final.pt está disponible y es EfficientNet,
# se usa como base. Si no, parte desde ImageNet directamente.
# ─────────────────────────────────────────────────────────────────────────────
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

modelo = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# Reemplazar clasificador final: 1000 clases ImageNet → 2 clases
in_features = modelo.classifier[1].in_features
modelo.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, 2)
)
modelo = modelo.to(device)

print('✅ EfficientNet-B0 listo (desde ImageNet)')
print(f'   Parámetros totales:    {sum(p.numel() for p in modelo.parameters()):,}')
print(f'   Parámetros entrenables: {sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 7 — Entrenamiento con early stopping y guardado del mejor modelo
#
# Mejoras vs versión anterior:
#   - 20 épocas (antes 10)
#   - Early stopping: para si val_acc no mejora en 5 épocas seguidas
#   - Guarda el mejor modelo (mayor val_acc), no el último
#   - ReduceLROnPlateau: baja lr automáticamente si se estanca
#   - Muestra mejor val_acc alcanzada
# ─────────────────────────────────────────────────────────────────────────────
import time

EPOCAS         = 20
PACIENCIA      = 5    # early stopping: parar si no mejora en N épocas
MEJOR_MODELO   = '/content/mejor_vehiculo.pth'

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

mejor_acc   = 0.0
sin_mejora  = 0

print('🚀 Iniciando entrenamiento...')
print(f'   {EPOCAS} épocas máx | Early stopping: {PACIENCIA} épocas sin mejora')
print('-' * 60)

start = time.time()
for epoch in range(EPOCAS):
    # ── Entrenamiento ────────────────────────────────────────────
    modelo.train()
    loss_total = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(modelo(imgs), labels)
        loss.backward()
        optimizer.step()
        loss_total += loss.item()

    # ── Validación ───────────────────────────────────────────────
    modelo.eval()
    correctos = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = modelo(imgs).argmax(dim=1)
            correctos += (preds == labels).sum().item()
            total     += labels.size(0)

    acc = correctos / total * 100
    scheduler.step(acc)

    # ── Guardar mejor modelo ─────────────────────────────────────
    mejoro = acc > mejor_acc
    if mejoro:
        mejor_acc = acc
        torch.save(modelo.state_dict(), MEJOR_MODELO)
        sin_mejora = 0
        marca = '⭐'
    else:
        sin_mejora += 1
        marca = ''

    print(f'Época {epoch+1:02d}/{EPOCAS} — '
          f'Loss: {loss_total/len(train_loader):.4f} | '
          f'Val Acc: {acc:.1f}% | '
          f'Mejor: {mejor_acc:.1f}% {marca}')

    # ── Early stopping ───────────────────────────────────────────
    if sin_mejora >= PACIENCIA:
        print(f'\n⏹️  Early stopping en época {epoch+1} (sin mejora por {PACIENCIA} épocas)')
        break

print('-' * 60)
print(f'🏆 Mejor val_acc: {mejor_acc:.1f}%')
print(f'⏱️  Tiempo total: {(time.time()-start)/60:.1f} minutos')
print(f'\n📦 Cargando mejor modelo guardado...')
modelo.load_state_dict(torch.load(MEJOR_MODELO))
print('✅ Mejor modelo cargado')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 8 — Evaluación detallada + matriz de confusión
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

modelo.eval()
clases = train_ds.classes   # ['infraccion', 'normal']
correctos_clase = defaultdict(int)
total_clase     = defaultdict(int)
confusion       = np.zeros((2, 2), dtype=int)

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = modelo(imgs).argmax(dim=1)
        for pred, label in zip(preds, labels):
            p, l = pred.item(), label.item()
            confusion[l][p] += 1
            total_clase[l]  += 1
            correctos_clase[l] += int(p == l)

print('=== PRECISIÓN POR CLASE ===')
for idx, nombre in enumerate(clases):
    if total_clase[idx] > 0:
        acc = correctos_clase[idx] / total_clase[idx] * 100
        print(f'  {nombre:12s}: {acc:.1f}%  ({correctos_clase[idx]}/{total_clase[idx]})')

total_global = sum(total_clase.values())
aciertos     = sum(correctos_clase.values())
print(f'\n  TOTAL        : {aciertos/total_global*100:.1f}%  ({aciertos}/{total_global})')

# Matriz de confusión
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(confusion, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(clases); ax.set_yticklabels(clases)
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(confusion[i][j]), ha='center', va='center',
                color='white' if confusion[i][j] > confusion.max()/2 else 'black', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 9 — Guardar modelo final + subir a Google Drive
#
# IMPORTANTE: El clasificador ahora es EfficientNet-B0, no ResNet-18.
# Debes actualizar vehiculo_estacionamiento_classifier.py en el proyecto
# para usar EfficientNet-B0 al cargar el modelo.
# ─────────────────────────────────────────────────────────────────────────────
import shutil

NOMBRE_FINAL = 'modelo_vehiculo_estacionamiento_v2.pth'

shutil.copy(MEJOR_MODELO, NOMBRE_FINAL)
print(f'✅ {NOMBRE_FINAL} guardado en /content/')

shutil.copy(NOMBRE_FINAL, f'/content/drive/MyDrive/{NOMBRE_FINAL}')
print(f'✅ Respaldado en Google Drive')

# Descargar directo al computador (descomenta si prefieres)
# from google.colab import files
# files.download(NOMBRE_FINAL)

print(f'\n📋 Resumen del modelo entrenado:')
print(f'   Arquitectura: EfficientNet-B0')
print(f'   Clases: infraccion=0, normal=1')
print(f'   Mejor val_acc: {mejor_acc:.1f}%')
print(f'   Archivo: {NOMBRE_FINAL}')
print(f'\n⚠️  Recuerda actualizar vehiculo_estacionamiento_classifier.py')
print(f'   para usar EfficientNet-B0 en lugar de ResNet-18')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 10 — Prueba con imágenes individuales
# ─────────────────────────────────────────────────────────────────────────────
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

def predecir(ruta_imagen):
    modelo.eval()
    img    = Image.open(ruta_imagen).convert('RGB')
    tensor = transform_val(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = modelo(tensor)
        probs  = F.softmax(logits, dim=1)[0]

    p_infraccion = probs[0].item() * 100
    p_normal     = probs[1].item() * 100
    clase_final  = clases[int(torch.argmax(probs).item())]

    print('-' * 40)
    print(f'🚗 Normal:     {p_normal:.1f}%')
    print(f'🚨 Infraccion: {p_infraccion:.1f}%')
    print('-' * 40)
    print(f'PREDICCIÓN: {clase_final.upper()}')

    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    color = 'red' if clase_final == 'infraccion' else 'green'
    plt.title(f'{clase_final.upper()} ({max(p_infraccion, p_normal):.1f}%)',
              fontsize=14, color=color, fontweight='bold')
    plt.axis('off')
    plt.show()

# Cambia 'test.jpg' por la imagen que quieres probar
# predecir('test.jpg')
print('ℹ️  Sube una imagen a Colab y llama: predecir("nombre.jpg")')